In [1]:
import numpy as np
from ase import Atoms
from pymatgen.core import Structure
from pymatgen.symmetry import analyzer

In [2]:
# revisit
def get_struct_wt_distortions(
    prist_stc: Structure, 
    rlxd_stc: Structure, 
    n_mupos: np.ndarray, 
    ipt_st: Structure
) -> Structure:
    """
    Experimental Function!

    Translates displacement due to the muon from one muon to a
    magnetically inequivalent site.
    Returns: Structure with translated displ and muon position

    This function assumes that H is the particle of interest.
    This is probably a problem when H atoms are already present.
    """
    tol = 0.001

    # get symmetry operations
    spp = analyzer.SpacegroupAnalyzer(ipt_st)
    ops = spp.get_symmetry_operations()
    # opsg = spp.get_space_group_operations()
    opg = spp.get_point_group_operations()

    # get  and remove relaxed muon position
    mupos_rlx = rlxd_stc.frac_coords[rlxd_stc.atomic_numbers.index(1)]
    # rlxd_stc.pop()
    rlxd_stc.remove_sites([rlxd_stc.atomic_numbers.index(1)])

    # remove initial muon position from prist_stc
    # prist_stc.pop()
    if 1 in prist_stc.atomic_numbers:
        prist_stc.remove_sites([prist_stc.atomic_numbers.index(1)])

    assert len(rlxd_stc.frac_coords) == len(prist_stc.frac_coords)

    # get the symmetry operations that can transform mupos_rlx to n_mupos
    symm_op = []
    for i, op in enumerate(opg):
        newp = op.operate(mupos_rlx) % 1
        if np.all(np.abs(newp - n_mupos) < tol):
            symm_op.append(i)

    nw_stc = prist_stc.copy()

    # get and transform displacement vectors
    if len(symm_op) > 0:
        disp = rlxd_stc.frac_coords - prist_stc.frac_coords
        t_disp = opg[symm_op[0]].operate_multi(disp)

        ##instead get  disp with transforming atoms
        # t_disp2 = np.zeros([nw_stc.num_sites, 3])
        for i in range(len(nw_stc)):
            nw_stc.translate_sites(i, t_disp[i], frac_coords=True, to_unit_cell=False)
            #
            # new_rlx_pos=ops[symm_op[0]].operate(rlxd_stc[i].frac_coords)
            # t_disp2[i] = new_rlx_pos - prist_stc[i].frac_coords
            # nw_stc.translate_sites(i, t_disp2[i],frac_coords=True, to_unit_cell=False)
    else:
        print(
            "Check symm op in get_struct_wt_distortions func, this should never happen"
        )

    try:
        nw_stc.append(
            species="H",
            coords=n_mupos,
            coords_are_cartesian=False,
            validate_proximity=True,
            properties={"kind_name": "H"},
        )
    except ValueError as e:
        print(e)
        return None
    
    return nw_stc

In [3]:
def get_distortions(
    unrelaxed_supercell: Atoms,
    relaxed_supercell: Atoms,
    muon_atomic_number: int = 1,
    only_final_muon_position: bool = True,
    verbose: bool = False,
    ) -> dict:
    """
    Get the distortions of the muon site in the relaxed supercell with respect to the unrelaxed supercell.  
    If use_final_muon_position is True, the final muon position is used to compute the distortions, i.e. is considered
    also to compute the mu-unrelaxed atoms distances. Otherwise, the initial muon position is used in the unrelaxed case: however
    this is less consistent, as the initial muon position is just the guess from the grid generated by the algorithm.
    """
    atm_indxes = [atom.index for atom in unrelaxed_supercell]
    mu_index = [atom.index for atom in unrelaxed_supercell if atom.number == muon_atomic_number]
    
    # updating the unrelaxed muon position to be the relaxed one, if only_final_muon_position is True:
    if only_final_muon_position:
        unrelaxed_supercell[mu_index[0]].position = relaxed_supercell[mu_index[0]].position
    
    unrelaxed_atm_dist = unrelaxed_supercell.get_distances(
        mu_index, atm_indxes, mic=True, vector=True
    )

    relaxed_atm_dist = relaxed_supercell.get_distances(
            mu_index, atm_indxes, mic=True, vector=True
    )
    
    distortion = (relaxed_atm_dist-unrelaxed_atm_dist)[:-1]
    
    unrelaxed_atm_dist_norm = np.linalg.norm(unrelaxed_atm_dist[:-1], axis=1)
    relaxed_atm_dist_norm = np.linalg.norm(relaxed_atm_dist[:-1], axis=1)
    
    # distortion is the modulus of the vector difference,
    # delta_distance is the difference of the norms.
    distortion_norm = np.linalg.norm(distortion, axis=1)
    delta_distance = relaxed_atm_dist_norm - unrelaxed_atm_dist_norm
    
    sort_distances = np.argsort(unrelaxed_atm_dist_norm)
    #distortion = distortion[sort_distances]
    
    sorted_atoms = [relaxed_supercell.get_chemical_symbols()[sorted_index] for sorted_index in sort_distances]
    
    distortions_dict = {}
    for element in set(relaxed_supercell.get_chemical_symbols()).difference({"H"}):
        distortions_dict[element] = {}
        indices = [i for x,i in zip(sorted_atoms, sort_distances) if x == element]
        distortions_dict[element]["atm_distance_init"] = unrelaxed_atm_dist_norm[indices]
        distortions_dict[element]["atm_distance_final"] = relaxed_atm_dist_norm[indices]
        distortions_dict[element]["distortion"] = distortion_norm[indices]
        distortions_dict[element]["delta_distance"] = delta_distance[indices]
        
    return distortions_dict